# Planning Time Analysis

Analyze planning_time.csv produced by planning-time runs.

## Section 1: Import Required Libraries

Load pandas and plotting helpers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
sns.set_theme(style='whitegrid')

## Section 2: Load and Clean Planning Time CSV

Read the CSV and normalize types.

In [ ]:
csv_path = Path('../runner/logs/planning_time.csv')
if not csv_path.exists():
    raise FileNotFoundError(f'Input file not found: {csv_path}')

df = pd.read_csv(csv_path, engine='python', on_bad_lines='skip')
raw_count = len(df)

if 'timestamp_utc' in df.columns:
    df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], errors='coerce', utc=True)
if 'planning_time_ms' in df.columns:
    df['planning_time_ms'] = pd.to_numeric(df['planning_time_ms'], errors='coerce')
if 'execution_time_ms' in df.columns:
    df['execution_time_ms'] = pd.to_numeric(df['execution_time_ms'], errors='coerce')
if 'exit_code' in df.columns:
    df['exit_code'] = pd.to_numeric(df['exit_code'], errors='coerce')
if 'run_index' in df.columns:
    df['run_index'] = pd.to_numeric(df['run_index'], errors='coerce')
    df = df[df['run_index'].fillna(-1) != 1]

print(f'Rows loaded: {raw_count}')
print(f'Rows after dropping run_index == 1: {len(df)}')
display(df.head())

df.info()

In [ ]:
missing_planning = df['planning_time_ms'].isna().sum() if 'planning_time_ms' in df.columns else len(df)
missing_execution = df['execution_time_ms'].isna().sum() if 'execution_time_ms' in df.columns else len(df)
failed_exit = (df['exit_code'].fillna(0) != 0).sum() if 'exit_code' in df.columns else 0

print(f'Missing planning_time_ms: {missing_planning}')
print(f'Missing execution_time_ms: {missing_execution}')
print(f'Non-zero exit_code: {failed_exit}')

## Section 3: Per-Query Planning Time Stats

Summaries using successful runs only.

In [ ]:
target_queries = ["APX1090", "APX1092", "APX1173", "PE002", "PE019"]
pattern = r"^(APX1090|APX1092|APX1173|PE002|PE019)"

if "test_name" not in df.columns:
    raise KeyError("test_name column not found in planning_time.csv")

filtered = df[df["test_name"].str.contains(pattern, regex=True, na=False)].copy()
print(f"Rows after filter: {len(filtered)}")
display(filtered["test_name"].value_counts())

valid = filtered.copy()
if "exit_code" in valid.columns:
    valid = valid[valid["exit_code"].fillna(0) == 0]
valid = valid[valid["planning_time_ms"].notna()]

def p95(x):
    return x.quantile(0.95)

def cv_pct(x):
    mean = x.mean()
    if mean == 0 or pd.isna(mean):
        return np.nan
    return x.std(ddof=1) / mean * 100

query_stats = (
    valid.groupby("test_name")["planning_time_ms"]
    .agg(["count", "mean", "median", "std", "min", "max", p95, cv_pct])
    .reset_index()
    .sort_values("median", ascending=False)
 )

exec_valid = filtered.copy()
if "exit_code" in exec_valid.columns:
    exec_valid = exec_valid[exec_valid["exit_code"].fillna(0) == 0]
if "execution_time_ms" in exec_valid.columns:
    exec_valid = exec_valid[exec_valid["execution_time_ms"].notna()]
    exec_query_stats = (
        exec_valid.groupby("test_name")["execution_time_ms"]
        .agg(["count", "mean", "median", "std", "min", "max", p95, cv_pct])
        .reset_index()
        .sort_values("median", ascending=False)
    )
else:
    exec_query_stats = pd.DataFrame()

display(query_stats)
if not exec_query_stats.empty:
    display(exec_query_stats)
else:
    print("execution_time_ms not available.")

## Section 3b: A/B Comparisons

Compare planning time for QueryA vs QueryB per base query.

In [ ]:
ab = valid.copy()
ab["base"] = ab["test_name"].str.extract(r"^(APX\d+|PE\d+)-query")
ab["variant"] = ab["test_name"].str.extract(r"-query([AB])", expand=False)
ab = ab.dropna(subset=["base", "variant"])

ab_stats = (
    ab.groupby(["base", "variant"])["planning_time_ms"]
    .agg(count="count", mean="mean", median="median", std="std")
    .reset_index()
)

a_stats = ab_stats[ab_stats["variant"] == "A"].rename(
    columns={"count": "count_A", "mean": "mean_A", "median": "median_A", "std": "std_A"}
)
b_stats = ab_stats[ab_stats["variant"] == "B"].rename(
    columns={"count": "count_B", "mean": "mean_B", "median": "median_B", "std": "std_B"}
)

ab_compare = a_stats.merge(b_stats, on="base", how="inner")
ab_compare["median_diff_A_minus_B"] = ab_compare["median_A"] - ab_compare["median_B"]
ab_compare["median_ratio_A_over_B"] = ab_compare["median_A"] / ab_compare["median_B"]
ab_compare["mean_diff_B_minus_A"] = ab_compare["mean_B"] - ab_compare["mean_A"]
ab_compare = ab_compare.sort_values("median_ratio_A_over_B", ascending=False)

exec_compare = pd.DataFrame()
exec_valid = filtered.copy()
if "exit_code" in exec_valid.columns:
    exec_valid = exec_valid[exec_valid["exit_code"].fillna(0) == 0]
if "execution_time_ms" in exec_valid.columns:
    exec_valid = exec_valid[exec_valid["execution_time_ms"].notna()]
    exec_ab = exec_valid.copy()
    exec_ab["base"] = exec_ab["test_name"].str.extract(r"^(APX\d+|PE\d+)-query")
    exec_ab["variant"] = exec_ab["test_name"].str.extract(r"-query([AB])", expand=False)
    exec_ab = exec_ab.dropna(subset=["base", "variant"])

    exec_stats = (
        exec_ab.groupby(["base", "variant"])["execution_time_ms"]
        .agg(count="count", mean="mean", median="median", std="std")
        .reset_index()
    )

    exec_a = exec_stats[exec_stats["variant"] == "A"].rename(
        columns={"count": "count_A", "mean": "mean_A", "median": "median_A", "std": "std_A"}
)
    exec_b = exec_stats[exec_stats["variant"] == "B"].rename(
        columns={"count": "count_B", "mean": "mean_B", "median": "median_B", "std": "std_B"}
)

    exec_compare = exec_a.merge(exec_b, on="base", how="inner")
    exec_compare["median_diff_A_minus_B"] = exec_compare["median_A"] - exec_compare["median_B"]
    exec_compare["median_ratio_A_over_B"] = exec_compare["median_A"] / exec_compare["median_B"]
    exec_compare["mean_diff_B_minus_A"] = exec_compare["mean_B"] - exec_compare["mean_A"]
    exec_compare = exec_compare.sort_values("median_ratio_A_over_B", ascending=False)

display(ab_compare)
if not exec_compare.empty:
    display(exec_compare)
else:
    print("execution_time_ms not available for A/B comparisons.")

## Section 3c: Statistical Tests (A vs B)

Nonparametric test per base query to check if A and B differ.

In [ ]:
try:
    from scipy.stats import mannwhitneyu
    _have_scipy = True
except Exception:
    _have_scipy = False

def permutation_test_median(a, b, n=5000, seed=42):
    rng = np.random.default_rng(seed)
    a = np.asarray(a)
    b = np.asarray(b)
    observed = abs(np.median(a) - np.median(b))
    combined = np.concatenate([a, b])
    count = 0
    for _ in range(n):
        rng.shuffle(combined)
        a_perm = combined[: len(a)]
        b_perm = combined[len(a) :]
        if abs(np.median(a_perm) - np.median(b_perm)) >= observed:
            count += 1
    return (count + 1) / (n + 1)

def _format_p_value(p_val: float, threshold: float = 1e-4) -> str:
    if pd.isna(p_val):
        return ""
    if p_val < threshold:
        return f"{p_val:.1e}"
    return f"{p_val:.4f}"

def run_ab_tests(df_in, metric):
    if metric not in df_in.columns:
        return pd.DataFrame()
    bases = sorted(df_in["test_name"].str.extract(r"^(APX\d+|PE\d+)-query").dropna()[0].unique())
    test_rows = []
    for base in bases:
        a_vals = df_in[df_in["test_name"] == f"{base}-queryA.sql"][metric].dropna()
        b_vals = df_in[df_in["test_name"] == f"{base}-queryB.sql"][metric].dropna()
        if len(a_vals) < 2 or len(b_vals) < 2:
            continue
        if _have_scipy:
            stat, p_value = mannwhitneyu(a_vals, b_vals, alternative="two-sided")
            method = "mannwhitneyu"
        else:
            stat = np.nan
            p_value = permutation_test_median(a_vals.values, b_vals.values)
            method = "perm_median"
        test_rows.append({
            "base": base,
            "n_A": len(a_vals),
            "n_B": len(b_vals),
            "median_A": float(np.median(a_vals)),
            "median_B": float(np.median(b_vals)),
            "median_diff_B_minus_A": float(np.median(b_vals) - np.median(a_vals)),
            "p_value": p_value,
            "p_value_display": _format_p_value(p_value),
            "method": method,
        })
    return pd.DataFrame(test_rows).sort_values("p_value")

plan_tests = run_ab_tests(valid, "planning_time_ms")
display(plan_tests)

exec_valid = filtered.copy()
if "exit_code" in exec_valid.columns:
    exec_valid = exec_valid[exec_valid["exit_code"].fillna(0) == 0]
if "execution_time_ms" in exec_valid.columns:
    exec_valid = exec_valid[exec_valid["execution_time_ms"].notna()]
exec_tests = run_ab_tests(exec_valid, "execution_time_ms")
if exec_tests.empty:
    print("execution_time_ms not available for statistical tests.")
else:
    display(exec_tests)

## Section 4: Per-Run Planning Time Stats

Compare planning time distribution by run_id for filtered queries.

In [ ]:
if "run_id" in valid.columns:
    run_stats = (
        valid.groupby("run_id")["planning_time_ms"]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .reset_index()
        .sort_values("median", ascending=False)
    )
    run_stats
else:
    print("run_id column not found.")

## Section 5: Visualizations

Histogram and per-query boxplots for the filtered queries.

In [ ]:
def plot_ab_box(df_in, metric, title, output_name):
    if metric not in df_in.columns:
        print(f"{metric} not available for boxplot.")
        return

    out_dir = Path("plots")
    out_dir.mkdir(parents=True, exist_ok=True)

    ab_plot = df_in.copy()
    ab_plot["base"] = ab_plot["test_name"].str.extract(r"^(APX\d+|PE\d+)-query")
    ab_plot["variant"] = ab_plot["test_name"].str.extract(r"-query([AB])", expand=False)
    ab_plot = ab_plot.dropna(subset=["base", "variant"])

    bases = sorted(ab_plot["base"].unique())
    data = []
    labels = []
    colors = []
    positions = []
    pos = 1

    for base in bases:
        a_vals = ab_plot[(ab_plot["base"] == base) & (ab_plot["variant"] == "A")][metric]
        b_vals = ab_plot[(ab_plot["base"] == base) & (ab_plot["variant"] == "B")][metric]
        if a_vals.empty or b_vals.empty:
            continue
        data.extend([a_vals.values, b_vals.values])
        labels.extend([f"{base} A", f"{base} B"])
        colors.extend(["#1f77b4", "#ff7f0e"])
        positions.extend([pos, pos + 1])
        pos += 3

    if not data:
        print("No A/B pairs available for boxplot.")
        return

    fig, ax = plt.subplots(figsize=(12, 5))
    bp = ax.boxplot(
        data,
        positions=positions,
        patch_artist=True,
        widths=0.6,
        showfliers=False,
    )
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_edgecolor("#333333")

    ax.set_title(title)
    ax.set_xlabel("Query")
    ax.set_ylabel("Time (ms)")
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    output_path = out_dir / f"{output_name}.svg"
    fig.savefig(output_path, format="svg")
    plt.show()

if valid.empty:
    print("No valid rows after filtering.")
else:
    plot_ab_box(valid, "planning_time_ms", "Planning Time by Query (A/B)", "planning_time_ab")
    exec_valid = filtered.copy()
    if "exit_code" in exec_valid.columns:
        exec_valid = exec_valid[exec_valid["exit_code"].fillna(0) == 0]
    if "execution_time_ms" in exec_valid.columns:
        exec_valid = exec_valid[exec_valid["execution_time_ms"].notna()]
    plot_ab_box(exec_valid, "execution_time_ms", "Execution Time by Query (A/B)", "execution_time_ab")